# Intracranial haemorrhage detection on head CT (EfficientNetB0)
Author: Abdullah Ghani, ECE Paris (Data & AI)

Binary classification (haemorrhage vs normal) of 2D head CT slices. Public dataset: Kaggle `felipekitamura/head-ct-hemorrhage` (200 slices, 100 per class).
Because the dataset is tiny, results are reported with **5-fold stratified cross-validation** (out-of-fold predictions, bootstrap 95% CI), not a single small test split.

> Research prototype. Not a medical device. Not for clinical use. Runtime: Colab GPU (T4).

In [ ]:
!pip install -q kagglehub
import os, glob, json, random, numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt, cv2
from tensorflow.keras import layers, models, optimizers
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
SEED=42; random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
import kagglehub
root=kagglehub.dataset_download("felipekitamura/head-ct-hemorrhage")
print(root); 
for r,d,f in os.walk(root): print(r, d[:3], f[:3])

In [ ]:
csv=glob.glob(root+"/**/labels.csv",recursive=True)[0]
lab=pd.read_csv(csv); lab.columns=[c.strip() for c in lab.columns]
print(lab.head(), lab.shape)
imgdir=os.path.dirname(glob.glob(root+"/**/000.png",recursive=True)[0])
idcol=lab.columns[0]; ycol=lab.columns[1]
files=np.array([os.path.join(imgdir,f"{int(str(i).split('.')[0]):03d}.png") for i in lab[idcol]])
y=lab[ycol].astype(int).values
print(len(files), np.bincount(y))
IMG=224
def load(p):
    im=cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB); return cv2.resize(im,(IMG,IMG))
X=np.stack([load(p) for p in files]).astype("float32")
print(X.shape)

In [ ]:
aug=models.Sequential([layers.RandomFlip("horizontal"),layers.RandomRotation(0.05),layers.RandomZoom(0.1),layers.RandomContrast(0.15)])
def build():
    base=tf.keras.applications.EfficientNetB0(include_top=False,weights="imagenet",input_shape=(IMG,IMG,3))
    base.trainable=False
    inp=layers.Input((IMG,IMG,3)); x=aug(inp); x=base(x,training=False)
    x=layers.GlobalAveragePooling2D()(x); x=layers.Dropout(0.4)(x); out=layers.Dense(1,activation="sigmoid")(x)
    return models.Model(inp,out), base
def fit(m, base, Xtr, ytr, Xva=None, yva=None):
    m.compile(optimizers.Adam(1e-3),"binary_crossentropy",metrics=["accuracy"])
    kw=dict(validation_data=(Xva,yva)) if Xva is not None else {}
    h1=m.fit(Xtr,ytr,epochs=15,batch_size=16,verbose=0,**kw)
    base.trainable=True
    for l in base.layers:
        if isinstance(l,layers.BatchNormalization): l.trainable=False
    m.compile(optimizers.Adam(1e-5),"binary_crossentropy",metrics=["accuracy"])
    h2=m.fit(Xtr,ytr,epochs=15,batch_size=16,verbose=0,**kw)
    return h1,h2
skf=StratifiedKFold(5,shuffle=True,random_state=SEED); oof=np.zeros(len(y))
for k,(tr,va) in enumerate(skf.split(X,y)):
    tf.keras.backend.clear_session(); m,b=build(); fit(m,b,X[tr],y[tr])
    oof[va]=m.predict(X[va],verbose=0).ravel()
    print("fold",k,"AUC",round(roc_auc_score(y[va],oof[va]),3),"acc",round(((oof[va]>.5)==y[va]).mean(),3))

In [ ]:
pred=(oof>0.5).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred).ravel()
def stats(yt,pp,ps):
    tn,fp,fn,tp=confusion_matrix(yt,pp,labels=[0,1]).ravel()
    return dict(acc=(tp+tn)/len(yt),sens=tp/max(tp+fn,1),spec=tn/max(tn+fp,1),auc=roc_auc_score(yt,ps))
pt=stats(y,pred,oof)
rng=np.random.default_rng(0); bs=[]
for _ in range(1000):
    i=rng.integers(0,len(y),len(y))
    if len(set(y[i]))<2: continue
    bs.append(stats(y[i],pred[i],oof[i]))
ci={k:[float(np.percentile([b[k] for b in bs],2.5)),float(np.percentile([b[k] for b in bs],97.5))] for k in pt}
res={"n_images":int(len(y)),"protocol":"5-fold stratified CV, out-of-fold predictions","point":{k:float(v) for k,v in pt.items()},"ci95_bootstrap":ci,"confusion_matrix":{"TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp)}}
json.dump(res,open("metrics.json","w"),indent=2); print(json.dumps(res,indent=2))

In [ ]:
cm=confusion_matrix(y,pred); fig,ax=plt.subplots(figsize=(4,4)); ax.imshow(cm,cmap="Blues")
for i in range(2):
    for j in range(2): ax.text(j,i,cm[i,j],ha="center",va="center")
ax.set_xticks([0,1]);ax.set_yticks([0,1]);ax.set_xticklabels(["normal","haemorrhage"]);ax.set_yticklabels(["normal","haemorrhage"]);ax.set_xlabel("predicted");ax.set_ylabel("true");ax.set_title("Out-of-fold confusion matrix")
plt.tight_layout(); plt.savefig("confusion_matrix.png",dpi=150); plt.show()
f,t,_=roc_curve(y,oof); plt.figure(figsize=(4,4)); plt.plot(f,t); plt.plot([0,1],[0,1],"--"); plt.xlabel("1 - specificity"); plt.ylabel("sensitivity"); plt.title(f"ROC (out-of-fold) AUC {pt['auc']:.3f}"); plt.tight_layout(); plt.savefig("roc_curve.png",dpi=150); plt.show()

In [ ]:
# Final model on all 200 slices (same recipe), saved for the demo
tf.keras.backend.clear_session(); m,b=build(); fit(m,b,X,y)
m.save("ich_effnetb0.keras")
# Example slices (training images, for illustration only)
fig,ax=plt.subplots(1,4,figsize=(12,3.2))
idx=list(np.where(y==1)[0][:2])+list(np.where(y==0)[0][:2])
for a,i in zip(ax,idx): a.imshow(X[i].astype("uint8")); a.set_title(("haemorrhage" if y[i] else "normal")+f"  p={m.predict(X[i:i+1],verbose=0)[0,0]:.2f} (train)"); a.axis("off")
plt.savefig("examples.png",dpi=130); plt.show()

In [ ]:
import zipfile
with zipfile.ZipFile("ich_results.zip","w") as z:
    for f in ["ich_effnetb0.keras","metrics.json","confusion_matrix.png","roc_curve.png","examples.png"]: z.write(f)
print("Saved ich_results.zip")

## Limitations
- Only 200 slices; confidence intervals are wide and the numbers must not be read as clinical performance.
- Per-slice classification with no patient identifiers, so slices from one patient may fall in different folds and cross-validation can be optimistic.
- Single source, no external validation, no haemorrhage subtype or localisation.
- The saved demo model is trained on all 200 slices; the reported metrics come from the cross-validation.